In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
import datetime
from datetime import date
from matplotlib import pyplot
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy
import pandas
from scipy.stats import ks_2samp, mannwhitneyu
import xarray as xr

In [2]:
# Define directories
Diri = '../ExtraTrack_Data/Output_Files_V7/'
Output_Diri = '../RCP_Figs/Analysis_Figs_V7.3.2/'

In [3]:
# Open file
def Open_File(File):
    DF = pandas.read_csv(File)
    DF = DF.drop("Unnamed: 0", axis=1)
    return (DF)

In [4]:
# Open each file
def Files_Open(Scenario, Diri, Subset):
    Data_DF = Open_File(Diri+Scenario+'_Data_'+Subset+'_Output.csv')
    ET_DF = Open_File(Diri+Scenario+'_ET_'+Subset+'_Output.csv')
    Codes_DF = Open_File(Diri+Scenario+'_Codes_Output.csv')
# Edit time format
    Time_Cols = ["ET Begin Time", "ET Complete Time", "Trop Peak Time", "Peak Time", "Genesis Time", "Final Time"]
    for Col in Time_Cols:
        ET_DF[Col] = pandas.to_datetime(ET_DF[Col], errors="coerce")
    Data_DF["Time(Z)"] = pandas.to_datetime(Data_DF["Time(Z)"], errors="coerce")
    return (Data_DF, ET_DF, Codes_DF)

In [5]:
# Create bins
def Create_Bins(Min, Max, Bin_Width):
    Bins = numpy.arange(Min, Max+Bin_Width, Bin_Width)
    return (Bins)
Lon_Bins = Create_Bins(-100,20,5)
Lat_Bins = Create_Bins(0,60,5)

In [6]:
# Number of years for each climate scenario
Num_Years = numpy.array([90,93,93])

In [7]:
# Open files
Control_Data, Control_ET, Control_Codes = Files_Open("Control", Diri, "SubsetB")
RCP45_Data, RCP45_ET, RCP45_Codes = Files_Open("RCP45", Diri, "SubsetB")
RCP85_Data, RCP85_ET, RCP85_Codes = Files_Open("RCP85", Diri, "SubsetB")

In [8]:
# Calculate statistical significance using KS Test
def KS_Test(Control_Array, RCP45_Array, RCP85_Array):
    P_Val_RCP45 = ks_2samp(Control_Array, RCP45_Array)[1]
    P_Val_RCP85 = ks_2samp(Control_Array, RCP85_Array)[1]
    return (P_Val_RCP45, P_Val_RCP85)

In [9]:
# Calculate statistical significance using Mann Whitney Test
def MW_Test(Control_Array, RCP45_Array, RCP85_Array):
    P_Val_RCP45 = mannwhitneyu(Control_Array, RCP45_Array)[1]
    P_Val_RCP85 = mannwhitneyu(Control_Array, RCP85_Array)[1]
    return (P_Val_RCP45, P_Val_RCP85)

In [10]:
# Create subsets for each storm phase
def Phase_Subsets(Data):
    Subset_0 = Data[Data["SLP(hPa)"] <= 1010].reset_index()
    Subset_Trop = Subset_0[Subset_0["Storm Phase"] == "Tropical"]
    Subset_Trans = Subset_0[(Subset_0["Storm Phase"] == "Transition")]
    Subset_Extra = Subset_0[Subset_0["Storm Phase"] == "Extratropical"]
    return (Subset_0, Subset_Trop, Subset_Trans, Subset_Extra)

In [11]:
# Find statistical significance for storm phase cumulative distributions
def Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, Var):
# Create subsets
    Control_All, Control_Trop, Control_Trans, Control_Extra = Phase_Subsets(Control_Data)
    RCP45_All, RCP45_Trop, RCP45_Trans, RCP45_Extra = Phase_Subsets(RCP45_Data)
    RCP85_All, RCP85_Trop, RCP85_Trans, RCP85_Extra = Phase_Subsets(RCP85_Data)
#
# Create empty arrays
    KS_P_Vals = numpy.zeros((2,4))
    MW_P_Vals = numpy.zeros((2,4))
#
# Find statistical significance for each phase
    for i in range(4):
        Control_Phase = Phase_Subsets(Control_Data)[i]
        RCP45_Phase = Phase_Subsets(RCP45_Data)[i]
        RCP85_Phase = Phase_Subsets(RCP85_Data)[i]
        KS_P_Vals[0][i], KS_P_Vals[1][i] = KS_Test(Control_Phase[Var], RCP45_Phase[Var], RCP85_Phase[Var])
        MW_P_Vals[0][i], MW_P_Vals[1][i] = MW_Test(Control_Phase[Var], RCP45_Phase[Var], RCP85_Phase[Var])
#
# Output as dataframe
    Signif_DF = pandas.DataFrame({"Variable": [Var, Var, Var, Var], \
                "Phase": ["All", "Tropical", "Transitioning", "Extratropical"], \
                "RCP4.5 (KS Test)": KS_P_Vals[0], "RCP8.5 (KS Test)": KS_P_Vals[1], \
                "RCP4.5 (MW Test)": MW_P_Vals[0], "RCP8.5 (MW Test)": MW_P_Vals[1]})
    Signif_DF = Signif_DF.round(3)
    return (Signif_DF)

In [12]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "SLP(hPa)")

,Variable,Phase,RCP4.5 (KS Test),RCP8.5 (KS Test),RCP4.5 (MW Test),RCP8.5 (MW Test)
0,SLP(hPa),All,0.000,0.000,0.002,0.0
1,SLP(hPa),Tropical,0.000,0.000,0.020,0.0
2,SLP(hPa),Transitioning,0.000,0.000,0.000,0.0
3,SLP(hPa),Extratropical,0.004,0.003,0.053,0.0


In [13]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "Lat")

,Variable,Phase,RCP4.5 (KS Test),RCP8.5 (KS Test),RCP4.5 (MW Test),RCP8.5 (MW Test)
0,Lat,All,0.000,0.00,0.000,0.000
1,Lat,Tropical,0.000,0.00,0.000,0.000
2,Lat,Transitioning,0.012,0.02,0.001,0.031
3,Lat,Extratropical,0.000,0.00,0.000,0.000


In [14]:
Phase_Diff_Signif(Control_Data, RCP45_Data, RCP85_Data, "Lon")

,Variable,Phase,RCP4.5 (KS Test),RCP8.5 (KS Test),RCP4.5 (MW Test),RCP8.5 (MW Test)
0,Lon,All,0.0,0.000,0.00,0.153
1,Lon,Tropical,0.0,0.000,0.00,0.695
2,Lon,Transitioning,0.0,0.529,0.02,0.599
3,Lon,Extratropical,0.0,0.000,0.00,0.000


In [15]:
Control_All, Control_Trop, Control_Trans, Control_Extra = Phase_Subsets(Control_Data)
RCP45_All, RCP45_Trop, RCP45_Trans, RCP45_Extra = Phase_Subsets(RCP45_Data)
RCP85_All, RCP85_Trop, RCP85_Trans, RCP85_Extra = Phase_Subsets(RCP85_Data)

In [16]:
numpy.median(Control_Trop["SLP(hPa)"]) - numpy.median(RCP45_Trop["SLP(hPa)"])

0.8100000000000591

In [17]:
numpy.median(Control_Trop["SLP(hPa)"]) - numpy.median(RCP85_Trop["SLP(hPa)"])

2.2799999999999727

In [18]:
len(Control_Trans[Control_Trans["SLP(hPa)"] <= 920]) / len(Control_Trans)

0.005344418052256532

In [19]:
len(RCP45_Trans[RCP45_Trans["SLP(hPa)"] <= 930]) / len(RCP45_Trans)

0.05767599660729432

In [20]:
len(RCP85_Trans[RCP85_Trans["SLP(hPa)"] <= 930]) / len(RCP85_Trans)

0.048986486486486486